## For EDA on Delay time departure performance

1. 'FilghtDate'
2. 'CRSDepTime'
3. 'DepDelay'
4. 'DepDel15'


## Import Libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1783596810834_0001,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.


## Create Spark Session

In [2]:
spark = SparkSession.builder \
    .appName("Flight_EDA") \
    .getOrCreate()

## Read Silver Layer

In [3]:
df = spark.read.parquet(
    "s3://airline-dataset-2020-2025/Silver/Flight_Data_2020_2025/"
)

## Verify Dataset

In [4]:
df.printSchema()

root
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: timestamp (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Operated_or_Branded_Code_Share_Partners: string (nullable = true)
 |-- DOT_ID_Marketing_Airline: integer (nullable = true)
 |-- IATA_Code_Marketing_Airline: string (nullable = true)
 |-- Flight_Number_Marketing_Airline: integer (nullable = true)
 |-- Originally_Scheduled_Code_Share_Airline: string (nullable = true)
 |-- DOT_ID_Originally_Scheduled_Code_Share_Airline: integer (nullable = true)
 |-- IATA_Code_Originally_Scheduled_Code_Share_Airline: string (nullable = true)
 |-- Flight_Num_Originally_Scheduled_Code_Share_Airline: integer (nullable = true)
 |-- Operating_Airline: string (nullable = true)
 |-- DOT_ID_Operating_Airline: integer (nullable = true)
 |-- IATA_Code_Operating_Airline: string (nullable = true)


In [5]:
df.count()

40910253

In [6]:
df.show(5, truncate=False)

+-------+-----+----------+---------+-------------------+-------------------------+---------------------------------------+------------------------+---------------------------+-------------------------------+---------------------------------------+----------------------------------------------+-------------------------------------------------+--------------------------------------------------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+--------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------+----------

In [7]:
df.show(5, vertical=True, truncate=False)

-RECORD 0-----------------------------------------------------------------
 Quarter                                            | 3                   
 Month                                              | 7                   
 DayofMonth                                         | 15                  
 DayOfWeek                                          | 4                   
 FlightDate                                         | 2021-07-15 00:00:00 
 Marketing_Airline_Network                          | AS                  
 Operated_or_Branded_Code_Share_Partners            | AS                  
 DOT_ID_Marketing_Airline                           | 19930               
 IATA_Code_Marketing_Airline                        | AS                  
 Flight_Number_Marketing_Airline                    | 669                 
 Originally_Scheduled_Code_Share_Airline            | null                
 DOT_ID_Originally_Scheduled_Code_Share_Airline     | null                
 IATA_Code_Originally_Sch

## Select Required Columns

In [8]:
dep_df = df.select(
    "FlightDate",
    "CRSDepTime",
    "DepDelay",
    "DepDel15"
)

## Check Data Types

In [9]:
dep_df.printSchema()

root
 |-- FlightDate: timestamp (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- DepDel15: double (nullable = true)

## Missing Values

In [10]:
from pyspark.sql.functions import col,isnan,when,count

dep_df.select([
    count(
        when(col(c).isNull(),c)
    ).alias(c)
    for c in dep_df.columns
]).show()

+----------+----------+--------+--------+
|FlightDate|CRSDepTime|DepDelay|DepDel15|
+----------+----------+--------+--------+
|         0|         0|  896518|  896518|
+----------+----------+--------+--------+

FlightDate - Fine, Every flight has a date.
CRSDepTime - Fine, Scheduled departure time is always available.
DepDelay - Fine, Mostly cancelled/diverted flights have no actual departure delay.
DepDel15 - Fine, Derived from DepDelay, so it's also missing when no departure occurred.

Why missing ?

In the Bureau of Transportation Statistics (BTS) dataset:

If a flight is cancelled, there is no actual departure, so DepDelay is NULL.
If a flight is diverted before normal completion, some delay fields can also be missing.
There may also be records with incomplete operational data.

This is normal and not a data error.

## Summary Statistics

In [11]:
dep_df.describe().show()

+-------+------------------+------------------+------------------+
|summary|        CRSDepTime|          DepDelay|          DepDel15|
+-------+------------------+------------------+------------------+
|  count|          40910253|          40013735|          40013735|
|   mean|1326.0078063315814|10.937542346396805|0.1890711027101069|
| stddev|484.86370190259487|53.013711054992775|0.3915650963273635|
|    min|                 1|            -131.0|               0.0|
|    max|              2400|            7223.0|               1.0|
+-------+------------------+------------------+------------------+

Count - Around 896,518 records have missing DepDelay and DepDel15, which matches your earlier null count.

Mean - Average scheduled departure is about 1:26 PM. Average departure delay is 10.94 minutes. About 18.9% of flights departed 15+ minutes late.

StdDev - Departure times are spread across the day. Delay times vary a lot.

Min - Earliest scheduled departure is 00:01. Some flights left 131 minutes early. DepDel15 minimum is correctly 0.

Max - Latest scheduled departure is 24:00 (midnight). Largest delay is 7,223 minutes (~120 hours), likely due to exceptional disruptions. DepDel15 maximum is correctly 1

Everything is reasonable except the very large maximum delay.

Maximum DepDelay = 7223 minutes
≈ 120 hours
≈ 5 days

Is rare but can occur in the BTS airline dataset during severe weather & operational disruptions.

## Delay Distribution

In [12]:
dep_df.groupBy("DepDel15").count().show()

+--------+--------+
|DepDel15|   count|
+--------+--------+
|     0.0|32448294|
|    null|  896518|
|     1.0| 7565441|
+--------+--------+

Out of 40.91 million flight records:

32.45 million (79.3%) departed on time or within 15 minutes.
7.57 million (18.5%) departed with a delay of 15 minutes or more.
896,518 (2.2%) records had missing departure delay indicators because they correspond to cancelled or diverted flights and were excluded from operational delay analysis.

## The raw data still contains cancelled/diverted flights, we need to remove them for departure delays.

In [13]:
from pyspark.sql.functions import col

dep_df = dep_df.filter(
    (col("Cancelled") == 0) &
    (col("Diverted") == 0)
)

In [14]:
dep_df.groupBy("DepDel15").count().show()

+--------+--------+
|DepDel15|   count|
+--------+--------+
|     0.0|32374352|
|     1.0| 7521027|
+--------+--------+

## Average Departure Delay

In [15]:
dep_df.select(
    avg("DepDelay")
).show()

+------------------+
|     avg(DepDelay)|
+------------------+
|10.859050292516335|
+------------------+

## Maximum Departure Delay

In [16]:
from pyspark.sql.functions import max

dep_df.select(
    max("DepDelay").alias("Maximum_Departure_Delay")
).show()

+-----------------------+
|Maximum_Departure_Delay|
+-----------------------+
|                 7223.0|
+-----------------------+

## Minimum Departure Delay (Early Departures)

In [17]:
from pyspark.sql.functions import min

dep_df.select(
    min("DepDelay").alias("Minimum_Departure_Delay")
).show()

+-----------------------+
|Minimum_Departure_Delay|
+-----------------------+
|                 -131.0|
+-----------------------+

## Delayed Flights (>15 Minutes)

In [18]:
from pyspark.sql.functions import col

dep_df.filter(
    col("DepDel15") == 1
).count()

7521027

## On-Time Flights

In [19]:
dep_df.filter(
    col("DepDel15") == 0
).count()

32374352

## Delay Percentage

In [20]:
total = dep_df.count()

delayed = dep_df.filter(col("DepDel15")==1).count()

ontime = dep_df.filter(col("DepDel15")==0).count()

print("Delayed % :", delayed*100/total)
print("On Time % :", ontime*100/total)

('Delayed % :', 18)
('On Time % :', 81)

## Daily Delay Trend

In [22]:
from pyspark.sql.functions import avg

daily_delay = (
    dep_df.groupBy("FlightDate")
          .agg(avg("DepDelay").alias("AverageDelay"))
          .orderBy("FlightDate")
)

In [24]:
daily_delay.show()

+-------------------+------------------+
|         FlightDate|      AverageDelay|
+-------------------+------------------+
|2020-01-01 00:00:00| 5.483573368868843|
|2020-01-02 00:00:00| 6.439283314669653|
|2020-01-03 00:00:00| 8.597145550020196|
|2020-01-04 00:00:00| 15.13856495391353|
|2020-01-05 00:00:00| 9.803447503917617|
|2020-01-06 00:00:00| 5.999514969795847|
|2020-01-07 00:00:00|2.3274583453722433|
|2020-01-08 00:00:00| 4.286567585363559|
|2020-01-09 00:00:00|1.3908347408658548|
|2020-01-10 00:00:00| 8.374550152256159|
|2020-01-11 00:00:00|15.255280131529025|
|2020-01-12 00:00:00| 9.998029741520853|
|2020-01-13 00:00:00| 8.472962577492194|
|2020-01-14 00:00:00| 4.884085487561462|
|2020-01-15 00:00:00| 2.541935180394186|
|2020-01-16 00:00:00|13.410246030675122|
|2020-01-17 00:00:00|12.279940407535562|
|2020-01-18 00:00:00|13.801413164745258|
|2020-01-19 00:00:00|    5.514187313663|
|2020-01-20 00:00:00| 5.020322610042832|
+-------------------+------------------+
only showing top

## Create Departure Hour

In [25]:
from pyspark.sql.functions import floor

dep_df = dep_df.withColumn(
    "DepartureHour",
    floor(col("CRSDepTime")/100)
)

## Flights by Departure Hour

In [36]:
hourly_flights = (
    dep_df.groupBy("DepartureHour")
          .count()
          .orderBy("DepartureHour")
)

hourly_flights.show(24)

+-------------+-------+
|DepartureHour|  count|
+-------------+-------+
|            0|  64253|
|            1|  22241|
|            2|   8145|
|            3|   5369|
|            4|   2755|
|            5| 942247|
|            6|2716746|
|            7|2754840|
|            8|2761855|
|            9|2348035|
|           10|2557870|
|           11|2559091|
|           12|2443865|
|           13|2428675|
|           14|2402952|
|           15|2348219|
|           16|2306413|
|           17|2524049|
|           18|2418213|
|           19|2101363|
|           20|1763074|
|           21|1242999|
|           22| 878577|
|           23| 293531|
+-------------+-------+
only showing top 24 rows

## Average Delay by Hour

In [37]:
hourly_delay = (
    dep_df.groupBy("DepartureHour")
          .agg(avg("DepDelay").alias("AverageDelay"))
          .orderBy("DepartureHour")
)

hourly_delay.show(24)

+-------------+------------------+
|DepartureHour|      AverageDelay|
+-------------+------------------+
|            0|12.378348092696061|
|            1|13.457803156332899|
|            2|12.762430939226519|
|            3| 11.96796423915068|
|            4| 8.871506352087115|
|            5| 4.602379206301532|
|            6|3.8293520999018678|
|            7| 4.326808453485502|
|            8|4.9412090786808145|
|            9| 5.922723894660855|
|           10| 7.150175732152142|
|           11| 8.084465929503875|
|           12| 9.559417561935705|
|           13|10.795365374123751|
|           14|12.309782301102977|
|           15|13.520065632719946|
|           16| 14.81062238202785|
|           17| 15.73998484181567|
|           18|16.889534544723727|
|           19|18.295096563516157|
|           20| 18.14506084259651|
|           21|18.313330903725586|
|           22|17.941779718795278|
|           23|14.120201954819082|
+-------------+------------------+
only showing top 24 

## Delayed Flights by Hour

In [28]:
from pyspark.sql.functions import sum

hourly_delayed = (
    dep_df.groupBy("DepartureHour")
          .agg(sum("DepDel15").alias("DelayedFlights"))
          .orderBy("DepartureHour")
)

hourly_delayed.show()

+-------------+--------------+
|DepartureHour|DelayedFlights|
+-------------+--------------+
|            0|       13908.0|
|            1|        4651.0|
|            2|        1982.0|
|            3|        1296.0|
|            4|         490.0|
|            5|       70969.0|
|            6|      208012.0|
|            7|      259488.0|
|            8|      308701.0|
|            9|      307519.0|
|           10|      381069.0|
|           11|      414484.0|
|           12|      435728.0|
|           13|      473670.0|
|           14|      510625.0|
|           15|      536660.0|
|           16|      561032.0|
|           17|      626623.0|
|           18|      640293.0|
|           19|      597884.0|
+-------------+--------------+
only showing top 20 rows

## Top 20 Worst Delay Days

In [29]:
worst_days = (
    daily_delay
    .orderBy(col("AverageDelay").desc())
    .limit(20)
)

worst_days.show()

+-------------------+------------------+
|         FlightDate|      AverageDelay|
+-------------------+------------------+
|2024-07-19 00:00:00| 69.54896034186253|
|2023-01-11 00:00:00| 65.14137445760525|
|2022-12-23 00:00:00|63.923519910106045|
|2022-12-24 00:00:00|61.474012474012476|
|2024-01-16 00:00:00|  49.5837922895358|
|2024-01-15 00:00:00| 48.95706455542022|
|2022-12-25 00:00:00| 44.32467013194722|
|2022-12-26 00:00:00|42.110869278459305|
|2025-12-07 00:00:00| 41.98774196601827|
|2025-02-16 00:00:00|41.600249519155795|
|2025-03-16 00:00:00| 40.91768407212622|
|2024-07-22 00:00:00| 40.53168677005552|
|2022-12-22 00:00:00| 39.31912159794417|
|2024-01-09 00:00:00|39.203521779425394|
|2025-11-30 00:00:00| 38.10186362639772|
|2022-03-12 00:00:00| 38.09609942168082|
|2024-07-21 00:00:00| 37.89873480037414|
|2022-01-02 00:00:00|37.404123356113246|
|2024-07-20 00:00:00|36.736979591836736|
|2025-12-14 00:00:00| 36.57538149507437|
+-------------------+------------------+

## Key Findings
Dataset contains over 40 million flight records.

Approximately 18–19% of flights departed at least 15 minutes late.

Departure delays vary by hour of the day.

Certain dates experience exceptionally high average departure delays, likely due to severe weather or operational disruptions.

## Classify Numerical & Categorical Columns

In [30]:
print("Numerical Columns")
print(["CRSDepTime","DepDelay","DepDel15"])

print("\nCategorical Columns")
print(["FlightDate"])

Numerical Columns
['CRSDepTime', 'DepDelay', 'DepDel15']

Categorical Columns
['FlightDate']

## Validate FlightDate Range

In [31]:
from pyspark.sql.functions import min,max

dep_df.select(
    min("FlightDate").alias("Start_Date"),
    max("FlightDate").alias("End_Date")
).show()

+-------------------+-------------------+
|         Start_Date|           End_Date|
+-------------------+-------------------+
|2020-01-01 00:00:00|2025-12-31 00:00:00|
+-------------------+-------------------+

## Count Unique Flight Dates

In [32]:
print(
"Total Operational Days:",
dep_df.select("FlightDate").distinct().count()
)

('Total Operational Days:', 2192)

## Validate CRSDepTime Values

In [33]:
dep_df.select(
    min("CRSDepTime").alias("Earliest"),
    max("CRSDepTime").alias("Latest")
).show()

+--------+------+
|Earliest|Latest|
+--------+------+
|       1|  2400|
+--------+------+

## Truthfulness Check (DepDel15)

In [34]:
dep_df.groupBy("DepDel15").count().show()

+--------+--------+
|DepDel15|   count|
+--------+--------+
|     0.0|32374352|
|     1.0| 7521027|
+--------+--------+

Only values are 0 and 1.
No invalid categories.

## Truthfulness Check (Negative Delay)

In [35]:
dep_df.filter(
    col("DepDelay") < 0
).show(20,False)

+-------------------+----------+--------+--------+-------------+
|FlightDate         |CRSDepTime|DepDelay|DepDel15|DepartureHour|
+-------------------+----------+--------+--------+-------------+
|2021-07-15 00:00:00|1955      |-7.0    |0.0     |19           |
|2021-07-15 00:00:00|2035      |-7.0    |0.0     |20           |
|2021-07-15 00:00:00|926       |-1.0    |0.0     |9            |
|2021-07-15 00:00:00|1530      |-3.0    |0.0     |15           |
|2021-07-15 00:00:00|700       |-9.0    |0.0     |7            |
|2021-07-15 00:00:00|1645      |-8.0    |0.0     |16           |
|2021-07-15 00:00:00|720       |-1.0    |0.0     |7            |
|2021-07-15 00:00:00|1950      |-8.0    |0.0     |19           |
|2021-07-15 00:00:00|822       |-6.0    |0.0     |8            |
|2021-07-15 00:00:00|800       |-2.0    |0.0     |8            |
|2021-07-15 00:00:00|1543      |-4.0    |0.0     |15           |
|2021-07-15 00:00:00|900       |-4.0    |0.0     |9            |
|2021-07-15 00:00:00|1945

## Validate KPI

Already done.


Average Delay

Delayed %

On-Time %

Maximum Delay

Minimum Delay



## EDA Summary

• Schema validated successfully.

• Data types verified.

• Null values analyzed and operational records filtered.

• Summary statistics generated.

• Daily delay trends analyzed.

• Hourly departure trends analyzed.

• Peak delay hours identified.

• Top 20 highest delay days identified.

• Time-series coverage validated.

EDA for assigned departure-related columns completed successfully.

## Since using EMR academic Pyspark kernel its not allowing to use pandas 5.20.1 ships with older packages, and the PySpark kernel often doesn't include pandas or matplotlib by default, So we'll skip this part as of now for doing later

## Plots 

Converting only aggregated results to pandas for it.

40 million rows × 4 columns will require several GBs of RAM.
Jupyter on EMR will likely crash with an OutOfMemory error.
Spark is designed to process large datasets distributed across the cluster.

In [39]:
!pip install pandas matplotlib 

invalid syntax (<stdin>, line 1)
  File "<stdin>", line 1
    !pip install pandas matplotlib
    ^
SyntaxError: invalid syntax



In [35]:
pdf = hourly_flights.toPandas()

Pandas >= 0.19.2 must be installed; however, it was not found.
Traceback (most recent call last):
  File "/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py", line 2075, in toPandas
    require_minimum_pandas_version()
  File "/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/utils.py", line 129, in require_minimum_pandas_version
    "it was not found." % minimum_pandas_version)
ImportError: Pandas >= 0.19.2 must be installed; however, it was not found.



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.bar(
    pdf["DepartureHour"],
    pdf["count"]
)

plt.title("Flights by Scheduled Departure Hour")

plt.xlabel("Hour")

plt.ylabel("Flights")

plt.show()

No module named matplotlib.pyplot
Traceback (most recent call last):
ImportError: No module named matplotlib.pyplot

